In [ ]:
DATA_DIR = "./drive/MyDrive/Project/nanoGPT/Kernel investigation/data"
FINEWEB_DIR = DATA_DIR + "/fineweb10B"
HELLASWAG_DIR = DATA_DIR + "/hellaswag"

In [ ]:
import json

def load_hellaswag(data_type:str):
    assert data_type in ["val","test"], 'data_type is either "val" or "test"'
    examples = []
    path = HELLASWAG_DIR+f"/hellaswag_{data_type}.jsonl"
    with open(path, "r") as f:
        for line in f:
            example = json.loads(line)   # one line = one JSON object
            examples.append(example)
    return examples

hella_swag_val = load_hellaswag("val")

In [ ]:
# range of label values
labels = []
for idx, data_sample in enumerate(hella_swag_val):
    if data_sample['label'] not in labels:
        print(data_sample['label'])
        labels.append(data_sample['label'])

# check whether the format differs or is fully consistent
criteria = set(hella_swag_val[0].keys())
for idx, data_sample in enumerate(hella_swag_val):
    if set(data_sample.keys()) != criteria:
        print(idx, data_sample)
        print("data format is different in ",idx)
        break

3
2
1
0


In [ ]:
hella_swag_val[0]

{'ind': 24,
 'activity_label': 'Roof shingle removal',
 'ctx_a': 'A man is sitting on a roof.',
 'ctx_b': 'he',
 'ctx': 'A man is sitting on a roof. he',
 'split': 'val',
 'split_type': 'indomain',
 'label': 3,
 'endings': ['is using wrap to wrap a pair of skis.',
  'is ripping level tiles off.',
  "is holding a rubik's cube.",
  'starts pulling up roofing on a roof.'],
 'source_id': 'activitynet~v_-JhWjGDPHMY'}

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")

In [ ]:
# Validation
tokenized_val = []
for sample in hella_swag_val:
    label = sample['label']
    tokenized_ctx = enc.encode_ordinary(sample['ctx'])
    ctx_len = len(tokenized_ctx)
    tokenized_endings = [tokenized_ctx + enc.encode_ordinary(" " + x) for x in sample['endings']]
    masking_info = [[0]*ctx_len + [1]*(len(x) - ctx_len) for x in tokenized_endings]
    tokenized_val.append({"label":label, "candidates":tokenized_endings, "masking_info": masking_info})

with open(HELLASWAG_DIR+"/tokenized_hellaswag_val.json", "w") as f:
    json.dump(tokenized_val, f)